In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

df_77 = pd.read_csv(r'C:\Nepal_Flood_Project\Data\Districts_77\final_flood_dataset_77districts.csv')
df_77['date'] = pd.to_datetime(df_77['date'])

feature_cols = [
    'temperature', 'rainfall', 'humidity', 'wind_speed',
    'discharge', 'rainfall_roll3', 'rainfall_roll7',
    'discharge_roll3', 'discharge_roll7',
    'month', 'location_encoded', 'terrain_encoded'
]

train_data = df_77[df_77['date'].dt.year <= 2020]
test_data  = df_77[df_77['date'].dt.year >= 2021]

X_train = train_data[feature_cols]
y_train = train_data['flood_risk_label']
X_test  = test_data[feature_cols]
y_test  = test_data['flood_risk_label']

print(f'Training: {len(X_train)} rows')
print(f'Testing:  {len(X_test)} rows')
print('\nTraining label counts:')
print(y_train.value_counts())

Training: 590667 rows
Testing:  140602 rows

Training label counts:
flood_risk_label
0    582965
1      7123
2       579
Name: count, dtype: int64


In [3]:
from imblearn.over_sampling import SMOTE

sampling_strategy = {
    1: 40000,   
    2: 15000,   
}

smote = SMOTE(random_state=42, k_neighbors=5, sampling_strategy=sampling_strategy)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('Before SMOTE:', y_train.value_counts().to_dict())
print('After SMOTE:', pd.Series(y_train_sm).value_counts().to_dict())

Before SMOTE: {0: 582965, 1: 7123, 2: 579}
After SMOTE: {0: 582965, 1: 40000, 2: 15000}


In [4]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0,
    n_jobs=-1
)
xgb_model.fit(X_train_sm, y_train_sm)

probs = xgb_model.predict_proba(X_test)
print('Training complete!')
print('Max High probability in test set:', probs[:, 2].max())

Training complete!
Max High probability in test set: 0.9577998


In [5]:
for threshold in [0.01, 0.02, 0.05, 0.10, 0.15]:
    custom_preds = np.where(
        probs[:, 2] > threshold, 2,
        np.argmax(probs[:, :2], axis=1)
    )
    print(f'\n--- Threshold = {threshold} ---')
    print(classification_report(y_test, custom_preds, target_names=['Low','Medium','High'], zero_division=0))
    


--- Threshold = 0.01 ---
              precision    recall  f1-score   support

         Low       0.98      0.93      0.95    136183
      Medium       0.31      0.04      0.08      4207
        High       0.02      0.93      0.04       212

    accuracy                           0.91    140602
   macro avg       0.44      0.64      0.36    140602
weighted avg       0.95      0.91      0.93    140602


--- Threshold = 0.02 ---
              precision    recall  f1-score   support

         Low       0.98      0.94      0.96    136183
      Medium       0.31      0.04      0.08      4207
        High       0.02      0.83      0.04       212

    accuracy                           0.92    140602
   macro avg       0.44      0.61      0.36    140602
weighted avg       0.95      0.92      0.93    140602


--- Threshold = 0.05 ---
              precision    recall  f1-score   support

         Low       0.97      0.96      0.97    136183
      Medium       0.31      0.04      0.08      42

In [7]:
from sklearn.preprocessing import LabelEncoder

le_location = LabelEncoder()
le_location.fit(df_77['location'])

le_terrain = LabelEncoder()
le_terrain.fit(df_77['terrain'])


check = pd.DataFrame({
    'location': df_77['location'],
    'existing_encoded': df_77['location_encoded'],
    'rebuilt_encoded': le_location.transform(df_77['location'])
})
print('Encoders match existing data?', (check['existing_encoded'] == check['rebuilt_encoded']).all())

Encoders match existing data? True


In [8]:
HIGH_RISK_THRESHOLD = 0.01

joblib.dump(xgb_model, r'C:\Nepal_Flood_Project\Data\Districts_77\flood_xgboost_model_77districts.pkl')
joblib.dump(HIGH_RISK_THRESHOLD, r'C:\Nepal_Flood_Project\Data\Districts_77\flood_xgboost_threshold_77districts.pkl')
joblib.dump(le_location, r'C:\Nepal_Flood_Project\Data\Districts_77\location_encoder_77districts.pkl')
joblib.dump(le_terrain, r'C:\Nepal_Flood_Project\Data\Districts_77\terrain_encoder_77districts.pkl')

print('Saved: model, threshold, and both encoders')

Saved: model, threshold, and both encoders


In [9]:
importance = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance)

rainfall_roll7      0.248446
terrain_encoded     0.129252
discharge_roll7     0.118506
discharge           0.101191
rainfall            0.093332
discharge_roll3     0.071876
month               0.064644
location_encoded    0.047204
temperature         0.043380
humidity            0.038134
rainfall_roll3      0.027614
wind_speed          0.016422
dtype: float32
